In [13]:
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image

from sklearn.metrics import mean_squared_error, r2_score
from sklearn.ensemble import HistGradientBoostingRegressor  # gradient boosting regressor. [web:227]

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import torchvision.models as models
from torchvision.models import ResNet18_Weights  # torchvision resnet18 weights API. [web:41]
from torchvision import transforms


In [14]:
proc = Path("data/processed")

X_train_tab = np.load(proc / "X_train_tab.npy")
X_val_tab   = np.load(proc / "X_val_tab.npy")
X_test_tab  = np.load(proc / "X_test_tab.npy")

y_train = np.load(proc / "y_train.npy")
y_val   = np.load(proc / "y_val.npy")

# Optional but recommended for stability
y_train_log = np.log1p(y_train)
y_val_log   = np.log1p(y_val)

train_ids = pd.read_csv(proc / "train_ids.csv")["id"].astype(str).values
val_ids   = pd.read_csv(proc / "val_ids.csv")["id"].astype(str).values
test_ids  = pd.read_csv(proc / "test_ids.csv")["id"].astype(str).values

device = "cuda" if torch.cuda.is_available() else "cpu"
device


'cpu'

In [15]:
def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))  # mean_squared_error is standard. [web:118]

def eval_price_space(y_true_log, y_pred_log):
    y_true = np.expm1(y_true_log)
    y_pred = np.expm1(y_pred_log)
    return rmse(y_true, y_pred), float(r2_score(y_true, y_pred))


In [16]:
tab_model = HistGradientBoostingRegressor(
    loss="squared_error",
    learning_rate=0.05,
    max_iter=2000,
    max_depth=None,
    max_leaf_nodes=64,
    min_samples_leaf=20,
    l2_regularization=0.0,
    early_stopping=True,
    random_state=42,
)  # estimator and params are documented. [web:227]

tab_model.fit(X_train_tab, y_train_log)

val_pred_log_tab = tab_model.predict(X_val_tab)
tab_rmse, tab_r2 = eval_price_space(y_val_log, val_pred_log_tab)

print({"tabular_only_rmse": tab_rmse, "tabular_only_r2": tab_r2})


{'tabular_only_rmse': 115791.03878671682, 'tabular_only_r2': 0.8931571298048744}


In [17]:
class ImageByIdDS(Dataset):
    def __init__(self, ids, images_dir, transform):
        self.ids = ids
        self.images_dir = Path(images_dir)
        self.transform = transform

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, i):
        pid = self.ids[i]
        p = self.images_dir / f"{pid}.png"
        img = Image.open(p).convert("RGB")
        return self.transform(img), pid


In [18]:
# Keep 256x256 tiles (no center crop) and normalize like ImageNet pretrained models. [web:41]
img_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

weights = ResNet18_Weights.DEFAULT
resnet = models.resnet18(weights=weights)  # weights param is documented. [web:41]

# Feature extractor: remove the classifier head -> outputs (B,512,1,1), then flatten to 512. [web:22]
feature_extractor = nn.Sequential(*list(resnet.children())[:-1]).to(device)
feature_extractor.eval()

# Usually best to not train this extractor at first
for p in feature_extractor.parameters():
    p.requires_grad = False

print("ready:", device)


ready: cpu


In [19]:
@torch.no_grad()
def compute_embeddings(ids, images_dir, batch_size=64, num_workers=2):
    ds = ImageByIdDS(ids, images_dir=images_dir, transform=img_transform)
    dl = DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=num_workers,
                    pin_memory=(device == "cuda"))

    all_emb = []
    all_ids = []

    for xb, pid in dl:
        xb = xb.to(device, non_blocking=True)
        feat = feature_extractor(xb).flatten(1)  # (B,512)
        all_emb.append(feat.cpu().numpy())
        all_ids.extend(list(pid))

    emb = np.concatenate(all_emb, axis=0)
    return np.array(all_ids, dtype=str), emb


In [20]:
train_img_dir = Path("data/images/train")
test_img_dir  = Path("data/images/test")

# Filter to only rows that have images (critical)
train_mask = np.array([(train_img_dir / f"{pid}.png").exists() for pid in train_ids])
val_mask   = np.array([(train_img_dir / f"{pid}.png").exists() for pid in val_ids])
test_mask  = np.array([(test_img_dir  / f"{pid}.png").exists() for pid in test_ids])

print("train images:", train_mask.sum(), "/", len(train_mask))
print("val images:",   val_mask.sum(),   "/", len(val_mask))
print("test images:",  test_mask.sum(),  "/", len(test_mask))

train_ids_mm = train_ids[train_mask]
val_ids_mm   = val_ids[val_mask]
test_ids_mm  = test_ids[test_mask]

X_train_tab_mm = X_train_tab[train_mask]
X_val_tab_mm   = X_val_tab[val_mask]
X_test_tab_mm  = X_test_tab[test_mask]

y_train_log_mm = y_train_log[train_mask]
y_val_log_mm   = y_val_log[val_mask]

# Compute embeddings
ids_tr, emb_tr = compute_embeddings(train_ids_mm, "data/images/train", batch_size=64)
ids_va, emb_va = compute_embeddings(val_ids_mm,   "data/images/train", batch_size=64)
ids_te, emb_te = compute_embeddings(test_ids_mm,  "data/images/test",  batch_size=64)

emb_tr.shape, emb_va.shape, emb_te.shape


train images: 12967 / 12967
val images: 3242 / 3242
test images: 5404 / 5404


((12967, 512), (3242, 512), (5404, 512))

In [21]:
X_train_fused = np.concatenate([X_train_tab_mm, emb_tr], axis=1)
X_val_fused   = np.concatenate([X_val_tab_mm,   emb_va], axis=1)

fused_model = HistGradientBoostingRegressor(
    loss="squared_error",
    learning_rate=0.05,
    max_iter=600,
    max_depth=None,
    max_leaf_nodes=128,
    min_samples_leaf=20,
    l2_regularization=0.0,
    early_stopping=True,
    random_state=42,
)  # estimator is documented. [web:227]

fused_model.fit(X_train_fused, y_train_log_mm)

val_pred_log_fused = fused_model.predict(X_val_fused)
fused_rmse, fused_r2 = eval_price_space(y_val_log_mm, val_pred_log_fused)

print({
    "tabular_only_rmse": tab_rmse,
    "tabular_only_r2": tab_r2,
    "tabular_plus_image_rmse": fused_rmse,
    "tabular_plus_image_r2": fused_r2,
})


{'tabular_only_rmse': 115791.03878671682, 'tabular_only_r2': 0.8931571298048744, 'tabular_plus_image_rmse': 123725.72093081522, 'tabular_plus_image_r2': 0.8780124154046486}


In [22]:
# Fit fused model on (train + val) for better final model
# First recompute embeddings for train+val (only those with images)
all_ids = np.concatenate([train_ids_mm, val_ids_mm])
all_X_tab = np.concatenate([X_train_tab_mm, X_val_tab_mm], axis=0)
all_y_log = np.concatenate([y_train_log_mm, y_val_log_mm], axis=0)

ids_all, emb_all = compute_embeddings(all_ids, "data/images/train", batch_size=64)
X_all_fused = np.concatenate([all_X_tab, emb_all], axis=1)

fused_model.fit(X_all_fused, all_y_log)

X_test_fused = np.concatenate([X_test_tab_mm, emb_te], axis=1)
test_pred_log = fused_model.predict(X_test_fused)
test_pred_price = np.expm1(test_pred_log)

out_dir = Path("outputs")
out_dir.mkdir(exist_ok=True)
sub = pd.DataFrame({"id": test_ids_mm, "predicted_price": test_pred_price})
sub.to_csv(out_dir / "predictions.csv", index=False)

sub.head(), out_dir / "predictions.csv"


(           id  predicted_price
 0  2591820310     3.617235e+05
 1  7974200820     8.763729e+05
 2  7701450110     1.149180e+06
 3  9522300010     2.101068e+06
 4  9510861140     6.666780e+05,
 PosixPath('outputs/predictions.csv'))